In [0]:
dbutils.widgets.text("Entity","demo","Please Enter Entity name")

In [0]:
table_name = dbutils.widgets.get("Entity")

In [0]:
from pyspark.sql.functions import col, explode_outer
from pyspark.sql.types import *

### Latest

In [0]:
schema_latest = StructType([
    StructField("results", ArrayType(
        StructType([
            StructField("datetime", StructType([
                StructField("utc", StringType()),
                StructField("local", StringType())
            ])),
            StructField("value", DoubleType()),
            StructField("coordinates", StructType([
                StructField("latitude", DoubleType()),
                StructField("longitude", DoubleType())
            ])),
            StructField("sensorsId", LongType()),
            StructField("locationsId", LongType())
        ])
    ))
])

In [0]:
df_latest_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(schema_latest)
    .load(f"s3://openaq-project-dm/raw/{table_name}/")
)

In [0]:
df_latest_stream = (
    df_latest_stream
    .withColumn("result", explode_outer(col("results")))
    .select(
        # Datetime
        col("result.datetime.utc").alias("datetime_utc"),
        col("result.datetime.local").alias("datetime_local"),

        # Value
        col("result.value").alias("value"),

        # Coordinates
        col("result.coordinates.latitude").alias("latitude"),
        col("result.coordinates.longitude").alias("longitude"),

        # IDs
        col("result.sensorsId").alias("sensor_id"),
        col("result.locationsId").alias("location_id"),
    )
)

In [0]:
(
    df_latest_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "s3://openaq-project-dm/raw/latest/checkpoints/")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"openaq.bronze.brz_{table_name}")
)

### Sensors

In [0]:
schema_Sensors = StructType([
    StructField("results", ArrayType(
        StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
            StructField("coverage", StructType([
                StructField("datetimeFrom", StructType([
                    StructField("local", StringType()),
                    StructField("utc", StringType())
                ])),
                StructField("datetimeTo", StructType([
                    StructField("local", StringType()),
                    StructField("utc", StringType())
                ])),
                StructField("expectedCount", LongType()),
                StructField("expectedInterval", StringType()),
                StructField("observedCount", LongType()),
                StructField("observedInterval", StringType()),
                StructField("percentComplete", DoubleType()),
                StructField("percentCoverage", DoubleType())
            ])),

            StructField("datetimeFirst", StructType([
                StructField("local", StringType()),
                StructField("utc", StringType())
            ])),
            StructField("datetimeLast", StructType([
                StructField("local", StringType()),
                StructField("utc", StringType())
            ])),
           
            StructField("latest", StructType([
                StructField("coordinates", StructType([
                    StructField("latitude", DoubleType()),
                    StructField("longitude", DoubleType())
                ])),
                StructField("datetime", StructType([
                    StructField("local", StringType()),
                    StructField("utc", StringType())
                ])),
                StructField("value", DoubleType())
            ])),

            StructField("parameter", StructType([
                StructField("displayName", StringType()),
                StructField("id", LongType()),
                StructField("name", StringType()),
                StructField("units", StringType())
            ])),

            StructField("summary", StructType([
                StructField("avg", DoubleType()),
                StructField("max", DoubleType()),
                StructField("median", StringType()),
                StructField("min", DoubleType()),
                StructField("q02", StringType()),
                StructField("q25", StringType()),
                StructField("q75", StringType()),
                StructField("q98", StringType()),
                StructField("sd", StringType())
            ]))
        ])
    ))
])

In [0]:
df_sensors_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(schema_Sensors)
    .load(f"s3://openaq-project-dm/raw/{table_name}/")
)

In [0]:
df_sensors_stream = (
    df_sensors_stream
    .withColumn("result", explode_outer(col("results")))
    .select(
        # Basic fields
        col("result.id").alias("sensor_id"),
        col("result.name").alias("sensor_name"),

        # Coverage
        col("result.coverage.datetimeFrom.local").alias("coverage_from_local"),
        col("result.coverage.datetimeFrom.utc").alias("coverage_from_utc"),
        col("result.coverage.datetimeTo.local").alias("coverage_to_local"),
        col("result.coverage.datetimeTo.utc").alias("coverage_to_utc"),
        col("result.coverage.expectedCount").alias("coverage_expected_count"),
        col("result.coverage.expectedInterval").alias("coverage_expected_interval"),
        col("result.coverage.observedCount").alias("coverage_observed_count"),
        col("result.coverage.observedInterval").alias("coverage_observed_interval"),
        col("result.coverage.percentComplete").alias("coverage_percent_complete"),
        col("result.coverage.percentCoverage").alias("coverage_percent_coverage"),

        # Datetime First & Last
        col("result.datetimeFirst.local").alias("datetime_first_local"),
        col("result.datetimeFirst.utc").alias("datetime_first_utc"),
        col("result.datetimeLast.local").alias("datetime_last_local"),
        col("result.datetimeLast.utc").alias("datetime_last_utc"),

        # Latest reading
        col("result.latest.coordinates.latitude").alias("latest_latitude"),
        col("result.latest.coordinates.longitude").alias("latest_longitude"),
        col("result.latest.datetime.local").alias("latest_datetime_local"),
        col("result.latest.datetime.utc").alias("latest_datetime_utc"),
        col("result.latest.value").alias("latest_value"),

        # Parameter
        col("result.parameter.id").alias("parameter_id"),
        col("result.parameter.name").alias("parameter_name"),
        col("result.parameter.displayName").alias("parameter_display_name"),
        col("result.parameter.units").alias("parameter_units"),

        # Summary stats
        col("result.summary.avg").alias("summary_avg"),
        col("result.summary.max").alias("summary_max"),
        col("result.summary.median").alias("summary_median"),
        col("result.summary.min").alias("summary_min"),
        col("result.summary.q02").alias("summary_q02"),
        col("result.summary.q25").alias("summary_q25"),
        col("result.summary.q75").alias("summary_q75"),
        col("result.summary.q98").alias("summary_q98"),
        col("result.summary.sd").alias("summary_sd"),
    )
)

In [0]:
(
    df_sensors_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "s3://openaq-project-dm/raw/Sensors/checkpoints/")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"openaq.bronze.brz_{table_name}")
)